# Crosswalk and Merge: 5Essentials → Community Areas

**Author:** An Nisa Astuti (annisaastuti@uchicago.edu)

**Group:** An Nisa Astuti and Yi Wang

This notebook creates a crosswalk between 5Essentials school-level data and Chicago's 77 community areas, then merges with 311, CPS, and ACS data for neighborhood-level analysis of educational inequality.


### The Problem: Mismatched Spatial Units

Our datasets exist at different spatial levels:

- **5Essentials** (school level, 648 schools) — Survey scores: Leaders, Teachers, Families, Environment, Instruction
- **CPS Progress Reports** (school level, ~650 schools/year) — Attendance, Graduation, College Enrollment
- **Chicago 311** (community area level, 77 areas) — Response times, closure rates
- **ACS Demographics** (community area level, 77 areas) — Poverty, income, unemployment

To analyze relationships across these datasets, we need to:
1. **Match each school to its community area** — Create a crosswalk linking school_id → community_area
2. **Aggregate school-level scores** — Calculate community area means from individual schools
3. **Merge all datasets** — Join on the common `community_area` key (1-77)


## Solution: Two-Step Matching

The 5Essentials data has no `community_area` column. The old CPS file (2011-12) has community area but only covers schools that existed then—many charter schools opened after 2012.

- **Step 1: Direct ID match (~77%)** — Join 5Essentials to old CPS by School_ID to get community_area
- **Step 2: Geodesic distance match (~23%)** — For newer schools, find the nearest reference school by coordinates and inherit its community_area

This works because schools in the same community area are physically close (Chicago's 77 community areas average ~3 km across). Average matching distance was 0.36 km.

## Data Sources

- `5essentials_schools_wide.csv` — Scraped 5Essentials survey data (648 schools, 2022-2024)
- `Chicago_Public_Schools_-_Progress_Report_Cards__2011-2012__20260218.csv` — Old CPS with community_area (for crosswalk)
- `Chicago_Public_Schools_-_School_Progress_Reports_SY2122_20260204.csv` — CPS outcomes 2021-22
- `Chicago_Public_Schools_-_School_Progress_Reports_SY2223_20260204.csv` — CPS outcomes 2022-23
- `Chicago_Public_Schools_-_School_Progress_Reports_SY2324_20260204.csv` — CPS outcomes 2023-24
- `Chicago_Public_Schools_-_School_Progress_Reports_SY2425_20260204.csv` — CPS outcomes 2024-25 (also used for coordinates)
- `311_metrics.csv` — 311 service request metrics by community area
- `master_acs_311_ca.csv` — ACS demographics by community area merged with 311

### AI Usage Disclosure

I used Chat-GPT assistance to understand GeoPy syntax, specifically that `geodesic()` takes `(lat, lon)` tuples and returns a distance object with a `.km` attribute. I also use Chat-GPT to debug coordinate order issues, discovering that GeoPy uses `(latitude, longitude)` order, which differs from some mapping libraries that use `(longitude, latitude)`. 

In [1]:
!pip install geopy

In [2]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import warnings
warnings.filterwarnings('ignore')


## 1. Geodesic Distance Function

In [3]:
def get_distance_km(lat1, lon1, lat2, lon2):
    """
    Calculate the geodesic distance between two points using GeoPy.
    
    Parameters:
    -----------
    lat1, lon1 : float
        Latitude and longitude of point 1
    lat2, lon2 : float
        Latitude and longitude of point 2
    
    Returns:
    --------
    float
        Distance in kilometers
    
    Reference:
    ----------
    GeoPy documentation: https://geopy.readthedocs.io/en/stable/#module-geopy.distance
    Uses Karney (2013) geodesic algorithm for WGS-84 ellipsoid.
    """
    # GeoPy takes coordinates as (latitude, longitude) tuples
    point1 = (lat1, lon1)
    point2 = (lat2, lon2)
    
    return geodesic(point1, point2).km

# Test 
# Distance from Lincoln Park to Loop (should be ~4.5 km)
dist = get_distance_km(41.9214, -87.6513, 41.8819, -87.6298)
print(f"Test: Lincoln Park to Loop = {dist:.2f} km")

Test: Lincoln Park to Loop = 4.74 km


## 2. Load Data

In [4]:

# 5Essentials data 
df_5e = pd.read_csv('5essentials_schools_wide.csv')
print(f"5Essentials: {len(df_5e)} schools")
print(f"  Columns: {df_5e.columns.tolist()[:8]}...")

# Old CPS Progress Reports (2011-12) - with community-area
df_old_cps = pd.read_csv('cps-data/Chicago_Public_Schools_-_Progress_Report_Cards_(2011-2012)_20260218.csv')
print(f"\nOld CPS (2011-12): {len(df_old_cps)} schools")

# Rename columns for consistency
df_old_cps = df_old_cps.rename(columns={
    'School ID': 'School_ID',
    'Community Area Number': 'community_area',
    'Community Area Name': 'community_area_name'
})

# New CPS Progress Reports (SY2425) - with coordinates
df_new_cps = pd.read_csv('cps-data/Chicago_Public_Schools_-_School_Progress_Reports_SY2425_20260204.csv')
print(f"New CPS (SY2425): {len(df_new_cps)} schools")

5Essentials: 648 schools
  Columns: ['school_id', 'school_name', 'short_name', 'school_type', 'address', 'student_response_rate', 'teacher_response_rate', 'parent_response_rate']...

Old CPS (2011-12): 566 schools
New CPS (SY2425): 649 schools


In [5]:
# Preview 5Essentials data
print("5Essentials sample:")
df_5e[['school_id', 'school_name', 'school_type', 'address']].head()

5Essentials sample:


,school_id,school_name,school_type,address
0,610229,A.N. Pritzker School,Elementary School (K/PK-8),"2009 W Schiller St, Chicago, IL 60622"
1,400172,ASPIRA Business and Finance,High School (9-12),"2989 N Milwaukee Ave, Chicago, IL 60618"
2,400013,ASPIRA Charter School - Early College High School,High School (9-12),"3986 W Barry Ave, Chicago, IL 60618"
3,400017,ASPIRA Charter School - Haugan Middle School,Middle School (6-8),"3729 W Leland Ave, Chicago, IL 60625"
4,610038,Abraham Lincoln Elementary School,Elementary School (K/PK-8),"615 W Kemper Pl, Chicago, IL 60614"


In [6]:
# Preview old CPS data (with community area)
print("Old CPS sample (has community_area):")
df_old_cps[['School_ID', 'Name of School', 'community_area', 'community_area_name', 'Latitude', 'Longitude']].head()

Old CPS sample (has community_area):


,School_ID,Name of School,community_area,community_area_name,Latitude,Longitude
0,610038,Abraham Lincoln Elementary School,7,LINCOLN PARK,41.924497,-87.644522
1,610281,Adam Clayton Powell Paideia Community Academy ...,43,SOUTH SHORE,41.760324,-87.556736
2,610185,Adlai E Stevenson Elementary School,70,ASHBURN,41.747111,-87.731702
3,609993,Agustin Lara Elementary Academy,61,NEW CITY,41.809757,-87.672145
4,610513,Air Force Academy High School,34,ARMOUR SQUARE,41.828146,-87.632794


## 3. Step 1: Direct Match by School_ID

First, we try to match 5Essentials schools directly to the old CPS data using School_ID. This works for schools that existed in 2011-12.

In [7]:

# 1. Direct match by school_ID

# Create lookup table from old CPS
ca_lookup = df_old_cps[['School_ID', 'community_area', 'community_area_name', 'Latitude', 'Longitude']].copy()
ca_lookup['School_ID'] = ca_lookup['School_ID'].astype(str)

# Prepare 5Essentials for matching
df_5e['school_id_str'] = df_5e['school_id'].astype(str)

# Direct merge
df_matched = df_5e.merge(
    ca_lookup[['School_ID', 'community_area', 'community_area_name']],
    left_on='school_id_str',
    right_on='School_ID',
    how='left'
)

# Count matches
direct_matched = df_matched['community_area'].notna().sum()
direct_unmatched = df_matched['community_area'].isna().sum()


print(f"  Matched: {direct_matched} ({direct_matched/len(df_5e)*100:.1f}%)")
print(f"  Unmatched: {direct_unmatched} ({direct_unmatched/len(df_5e)*100:.1f}%)")

  Matched: 498 (76.9%)
  Unmatched: 150 (23.1%)


In [8]:
# Sample of unmatched schools
unmatched_schools = df_matched[df_matched['community_area'].isna()][['school_id', 'school_name', 'school_type']]
print(f"Sample unmatched schools (newer schools not in 2011-12 data):")
unmatched_schools.head(10)

Sample unmatched schools (newer schools not in 2011-12 data):


,school_id,school_name,school_type
1,400172,ASPIRA Business and Finance,High School (9-12)
2,400013,ASPIRA Charter School - Early College High School,High School (9-12)
3,400017,ASPIRA Charter School - Haugan Middle School,Middle School (6-8)
5,400009,Academy for Global Citizenship Charter School,Elementary School (K/PK-8)
6,400081,Acero Charter Schools - Bartolome de las Casas,Elementary School (K/PK-8)
7,400153,Acero Charter Schools - Brighton Park,Elementary School (K/PK-8)
8,400082,Acero Charter Schools - Carlos Fuentes,Elementary School (K/PK-8)
9,400114,Acero Charter Schools - Esmeralda Santiago,Elementary School (K/PK-8)
10,400112,Acero Charter Schools - Jovita Idar,Elementary School (K/PK-8)
11,400085,Acero Charter Schools - Major Hector P. Garcia MD,High School (9-12)


## 4. Step 2: Geodesic Matching for Unmatched Schools

For schools that don't have a direct match (mostly charter schools opened after 2012), we use coordinates to find the nearest school in the old CPS data and inherit its community area.

In [9]:

# 2. Geodesic matching

# Get coordinates from new CPS file
coords_new = df_new_cps[['School_ID', 'School_Latitude', 'School_Longitude', 'Short_Name']].copy()
coords_new['School_ID'] = coords_new['School_ID'].astype(str)

# Build reference table: old CPS schools with coordinates
ref_schools = df_old_cps[['School_ID', 'Latitude', 'Longitude', 'community_area', 'community_area_name']].copy()
ref_schools = ref_schools.dropna(subset=['Latitude', 'Longitude'])
ref_schools['School_ID'] = ref_schools['School_ID'].astype(str)

print(f"Reference schools with coordinates: {len(ref_schools)}")

Reference schools with coordinates: 566


In [10]:
# Get list of unmatched school IDs
unmatched_mask = df_matched['community_area'].isna()
unmatched_ids = df_matched.loc[unmatched_mask, 'school_id_str'].tolist()

print(f"Schools to match via geodesic distance: {len(unmatched_ids)}")

Schools to match via geodesic distance: 150


In [11]:
# Geodesic mnatching loop using Geopy

geodesic_matches = []

for i, sid in enumerate(unmatched_ids):
    # Get coordinates from new CPS file
    coord_row = coords_new[coords_new['School_ID'] == sid]
    
    # If no coordinates found, mark as unmatched
    if len(coord_row) == 0 or pd.isna(coord_row.iloc[0]['School_Latitude']):
        geodesic_matches.append({
            'school_id': sid,
            'community_area': np.nan,
            'community_area_name': np.nan,
            'match_method': 'no_coords',
            'nearest_school_id': np.nan,
            'distance_km': np.nan
        })
        continue
    
    school_lat = coord_row.iloc[0]['School_Latitude']
    school_lon = coord_row.iloc[0]['School_Longitude']
    
    # Find nearest school in reference data
    min_dist = float('inf')
    best_ca = np.nan
    best_name = np.nan
    best_ref_id = np.nan
    
    for _, ref in ref_schools.iterrows():
        # Using GeoPy geodesic distance
        dist = get_distance_km(school_lat, school_lon, ref['Latitude'], ref['Longitude'])
        if dist < min_dist:
            min_dist = dist
            best_ca = ref['community_area']
            best_name = ref['community_area_name']
            best_ref_id = ref['School_ID']
    
    geodesic_matches.append({
        'school_id': sid,
        'community_area': best_ca,
        'community_area_name': best_name,
        'match_method': 'geodesic',
        'nearest_school_id': best_ref_id,
        'distance_km': min_dist
    })
    
    # Progress indicator
    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1} / {len(unmatched_ids)} schools...")

df_geodesic = pd.DataFrame(geodesic_matches)
print(df_geodesic)

  Processed 50 / 150 schools...
  Processed 100 / 150 schools...
  Processed 150 / 150 schools...
    school_id  community_area community_area_name match_method  \
0      400172            21.0            AVONDALE     geodesic   
1      400013            21.0            AVONDALE     geodesic   
2      400017            14.0         ALBANY PARK     geodesic   
3      400009            56.0      GARFIELD RIDGE     geodesic   
4      400081            31.0     LOWER WEST SIDE     geodesic   
..        ...             ...                 ...          ...   
145    400139            46.0       SOUTH CHICAGO     geodesic   
146    400141             3.0              UPTOWN     geodesic   
147    400144            25.0              AUSTIN     geodesic   
148    400143            23.0       HUMBOLDT PARK     geodesic   
149    400145            35.0             DOUGLAS     geodesic   

    nearest_school_id  distance_km  
0              610144     0.609309  
1              610541     0.359710 

In [12]:
# Geodesic match results

geodesic_ok = df_geodesic['community_area'].notna().sum()
geodesic_fail = df_geodesic['community_area'].isna().sum()

print(f"  Matched via geodesic: {geodesic_ok}")
print(f"  Still unmatched (no coords): {geodesic_fail}")

if geodesic_ok > 0:
    print(f"\n  Distance statistics:")
    print(f"    Mean: {df_geodesic['distance_km'].mean():.3f} km")
    print(f"    Max:  {df_geodesic['distance_km'].max():.3f} km")
    print(f"    Min:  {df_geodesic['distance_km'].min():.3f} km")

print("Geodesic matches:")
df_geodesic[df_geodesic['match_method'] == 'geodesic'].head(10)

  Matched via geodesic: 147
  Still unmatched (no coords): 3

  Distance statistics:
    Mean: 0.363 km
    Max:  0.958 km
    Min:  0.003 km
Geodesic matches:


,school_id,community_area,community_area_name,match_method,nearest_school_id,distance_km
0,400172,21.0,AVONDALE,geodesic,610144,0.609309
1,400013,21.0,AVONDALE,geodesic,610541,0.359710
2,400017,14.0,ALBANY PARK,geodesic,609972,0.285794
3,400009,56.0,GARFIELD RIDGE,geodesic,609981,0.475691
4,400081,31.0,LOWER WEST SIDE,geodesic,609867,0.387113
5,400153,58.0,BRIGHTON PARK,geodesic,610174,0.430954
6,400082,21.0,AVONDALE,geodesic,610039,0.376622
7,400114,24.0,WEST TOWN,geodesic,609759,0.356596
8,400112,63.0,GAGE PARK,geodesic,610157,0.500229
9,400085,57.0,ARCHER HEIGHTS,geodesic,609903,0.559771


## 5. Step 3: Combine Results into Final Crosswalk

In [13]:
# Combine results

# Add match_method column for direct matches
df_matched['match_method'] = 'direct'
df_matched.loc[unmatched_mask, 'match_method'] = 'pending'

# Update unmatched rows with geodesic results
for _, row in df_geodesic.iterrows():
    mask = df_matched['school_id_str'] == row['school_id']
    df_matched.loc[mask, 'community_area'] = row['community_area']
    df_matched.loc[mask, 'community_area_name'] = row['community_area_name']
    df_matched.loc[mask, 'match_method'] = row['match_method']

print(df_matched)

     school_id                                        school_name  \
0       610229                               A.N. Pritzker School   
1       400172                        ASPIRA Business and Finance   
2       400013  ASPIRA Charter School - Early College High School   
3       400017       ASPIRA Charter School - Haugan Middle School   
4       610038                  Abraham Lincoln Elementary School   
..         ...                                                ...   
643     400139               YCCS - Sullivan House Alternative HS   
644     400141                    YCCS - Truman Middle College HS   
645     400144                                     YCCS - West HS   
646     400143               YCCS - West Town Acad Alternative HS   
647     400145         YCCS - Youth Connection Leadership Acad HS   

                         short_name                 school_type  \
0                          Pritzker  Elementary School (K/PK-8)   
1     ASPIRA - Business and Finance  

In [14]:
# Final crosswalk summary

final_matched = df_matched['community_area'].notna().sum()
final_unmatched = df_matched['community_area'].isna().sum()

print(f"\nTotal schools: {len(df_matched)}")
print(f"  Matched: {final_matched} ({final_matched/len(df_matched)*100:.1f}%)")
print(f"  Unmatched: {final_unmatched}")

print(f"\nBy match method:")
print(df_matched['match_method'].value_counts().to_string())

print(f"\nCommunity areas covered: {df_matched['community_area'].nunique()}")


Total schools: 648
  Matched: 645 (99.5%)
  Unmatched: 3

By match method:
match_method
direct       498
geodesic     147
no_coords      3

Community areas covered: 77


In [15]:
# Unmatched schools
if final_unmatched > 0:
    print(f"\nUnmatched schools ({final_unmatched}):")
    display(df_matched[df_matched['community_area'].isna()][['school_id', 'school_name', 'school_type']])


Unmatched schools (3):


,school_id,school_name,school_type
93,4001151,Catalyst Maria Charter Elementary School,Elementary School (K/PK-8)
94,400182,Catalyst Maria Charter High School,High School (9-12)
536,910572,Safe Achieve Academy ES and HS,High School (9-12)


In [16]:
# Final crosswalk

crosswalk = df_matched[['school_id', 'school_name', 'school_type', 
                        'community_area', 'community_area_name', 'match_method']].copy()

# Convert community_area to int (where not null)
crosswalk['community_area'] = crosswalk['community_area'].apply(
    lambda x: int(x) if pd.notna(x) else np.nan
)

# Save crosswalk
crosswalk.to_csv('5essentials_community_area_crosswalk.csv', index=False)

# Sample
crosswalk.head(10)

,school_id,school_name,school_type,community_area,community_area_name,match_method
0,610229,A.N. Pritzker School,Elementary School (K/PK-8),24.0,WEST TOWN,direct
1,400172,ASPIRA Business and Finance,High School (9-12),21.0,AVONDALE,geodesic
2,400013,ASPIRA Charter School - Early College High School,High School (9-12),21.0,AVONDALE,geodesic
3,400017,ASPIRA Charter School - Haugan Middle School,Middle School (6-8),14.0,ALBANY PARK,geodesic
4,610038,Abraham Lincoln Elementary School,Elementary School (K/PK-8),7.0,LINCOLN PARK,direct
5,400009,Academy for Global Citizenship Charter School,Elementary School (K/PK-8),56.0,GARFIELD RIDGE,geodesic
6,400081,Acero Charter Schools - Bartolome de las Casas,Elementary School (K/PK-8),31.0,LOWER WEST SIDE,geodesic
7,400153,Acero Charter Schools - Brighton Park,Elementary School (K/PK-8),58.0,BRIGHTON PARK,geodesic
8,400082,Acero Charter Schools - Carlos Fuentes,Elementary School (K/PK-8),21.0,AVONDALE,geodesic
9,400114,Acero Charter Schools - Esmeralda Santiago,Elementary School (K/PK-8),24.0,WEST TOWN,geodesic


## 6. Aggregate 5Essentials to Community Area Level

Now we aggregate school-level 5Essentials scores to community area level by taking the mean of all schools in each community area.

In [17]:
# Compute 3-year average scores

# Define the 5 Essentials
essentials = ['leaders', 'teachers', 'families', 'environment', 'instruction']
years = ['2022', '2023', '2024']

# For each essential, compute 3-year average
for ess in essentials:
    score_cols = [f'essential_{ess}_score_{yr}' for yr in years]
    
    # Replace invalid values (-1000, -1100, -1200) with NaN
    for col in score_cols:
        if col in df_matched.columns:
            df_matched[col] = df_matched[col].replace([-1000, -1100, -1200, -1300], np.nan)
    
    # Compute 3-year average
    df_matched[f'avg_{ess}_score'] = df_matched[score_cols].mean(axis=1, skipna=True)
    
df_matched[[f'avg_{ess}_score' for ess in essentials]].describe()

,avg_leaders_score,avg_teachers_score,avg_families_score,avg_environment_score,avg_instruction_score
count,643.000000,643.000000,629.000000,632.000000,631.000000
mean,51.911353,50.948419,52.071807,49.299314,54.422345
std,11.628383,10.868804,12.106854,11.872831,10.535740
min,16.000000,19.000000,18.000000,19.000000,29.333333
25%,44.333333,43.666667,44.000000,41.666667,47.583333
50%,52.000000,50.666667,52.000000,47.333333,52.666667
75%,59.666667,57.666667,59.666667,55.000000,59.666667
max,94.333333,94.000000,99.000000,99.000000,97.666667


In [18]:
# Aggregate to community area level

# Filter to matched schools only
df_for_agg = df_matched[df_matched['community_area'].notna()].copy()

# Define aggregation
avg_cols = [f'avg_{ess}_score' for ess in essentials]

agg_dict = {'school_id': 'count'}  # Count of schools
for col in avg_cols:
    agg_dict[col] = 'mean'  # Mean of scores

# Aggregate
df_5e_by_ca = df_for_agg.groupby(['community_area', 'community_area_name']).agg(agg_dict).reset_index()

# Rename columns
df_5e_by_ca = df_5e_by_ca.rename(columns={'school_id': 'n_schools_5e'})

# Sort by community area
df_5e_by_ca = df_5e_by_ca.sort_values('community_area').reset_index(drop=True)

df_5e_by_ca.head(10)

,community_area,community_area_name,n_schools_5e,avg_leaders_score,avg_teachers_score,avg_families_score,avg_environment_score,avg_instruction_score
0,1.0,ROGERS PARK,8,41.166667,36.833333,43.166667,47.958333,46.875000
1,2.0,WEST RIDGE,9,60.296296,58.777778,59.555556,54.333333,56.481481
2,3.0,UPTOWN,6,51.777778,49.305556,47.361111,52.777778,54.305556
3,4.0,LINCOLN SQUARE,5,49.333333,51.000000,64.333333,52.466667,53.333333
4,5.0,NORTH CENTER,7,51.404762,51.190476,59.583333,61.761905,53.476190
5,6.0,LAKE VIEW,11,49.666667,51.257576,61.954545,53.590909,54.621212
6,7.0,LINCOLN PARK,8,50.958333,51.145833,63.875000,53.190476,58.761905
7,8.0,NEAR NORTH SIDE,7,47.142857,47.619048,54.285714,50.023810,54.547619
8,9.0,EDISON PARK,3,66.166667,64.166667,67.333333,66.166667,53.833333
9,10.0,NORWOOD PARK,8,55.395833,55.041667,62.729167,53.095238,53.904762


In [19]:
# Save aggregated 5Essentials
df_5e_by_ca.to_csv('5essentials_by_community_area.csv', index=False)

## 7. Load and Merge All Datasets

Now we merge the aggregated 5Essentials data with 311 metrics, CPS progress reports, and ACS demographics.

In [20]:
# Load aggregated datasets
df_5e_ca = pd.read_csv('5essentials_by_community_area.csv')
df_cps_ca = pd.read_csv('cps_by_community_area.csv')
df_311_raw = pd.read_csv('311_metrics.csv')
df_acs = pd.read_csv('master_acs_311_ca.csv')

print(f"5Essentials: {len(df_5e_ca)} community areas")
print(f"CPS: {len(df_cps_ca)} community areas")
print(f"311 raw: {len(df_311_raw)} rows (multiple years)")
print(f"ACS: {len(df_acs)} community areas")

5Essentials: 77 community areas
CPS: 77 community areas
311 raw: 1394 rows (multiple years)
ACS: 77 community areas


In [21]:
# Aggregate 311 data to community area level
df_311_ca = df_311_raw.groupby('community_area').agg({
    'n_requests': 'sum',
    'n_closed': 'sum',
    'share_closed': 'mean',
    'median_ttc_hours': 'mean',
    'p75_ttc_hours': 'mean',
    'pct_closed_24h': 'mean',
    'pct_closed_7d': 'mean',
    'pct_closed_30d': 'mean'
}).reset_index()

# Rename to match final output
df_311_ca = df_311_ca.rename(columns={
    'n_requests': 'sr_total_requests',
    'n_closed': 'sr_total_closed',
    'share_closed': 'sr_avg_share_closed',
    'median_ttc_hours': 'sr_avg_median_ttc_hours',
    'p75_ttc_hours': 'sr_avg_p75_ttc_hours',
    'pct_closed_24h': 'sr_avg_pct_closed_24h',
    'pct_closed_7d': 'sr_avg_pct_closed_7d',
    'pct_closed_30d': 'sr_avg_pct_closed_30d'
})

print(f"311 aggregated: {len(df_311_ca)} community areas")

311 aggregated: 77 community areas


In [22]:
# Prepare ACS data (select only ACS columns)
acs_cols = [
    'community_area', 'total_population', 'poverty_rate', 
    'median_household_income_weighted', 'unemployment_rate', 
    'pct_less_than_hs', 'pct_bachelor_plus', 'snap_rate', 
    'public_assist_rate', 'rent_burden_30p_rate', 'rent_burden_50p_rate', 
    'crowding_rate', 'pct_renter', 'vacancy_rate', 'no_vehicle_rate', 
    'pct_white_non_hispanic', 'pct_black_non_hispanic', 'pct_hispanic'
]

df_acs_ca = df_acs[acs_cols].copy()
df_acs_ca.columns = ['community_area'] + ['acs_' + c for c in df_acs_ca.columns[1:]]
print(f"ACS columns: {df_acs_ca.columns.tolist()}")

ACS columns: ['community_area', 'acs_total_population', 'acs_poverty_rate', 'acs_median_household_income_weighted', 'acs_unemployment_rate', 'acs_pct_less_than_hs', 'acs_pct_bachelor_plus', 'acs_snap_rate', 'acs_public_assist_rate', 'acs_rent_burden_30p_rate', 'acs_rent_burden_50p_rate', 'acs_crowding_rate', 'acs_pct_renter', 'acs_vacancy_rate', 'acs_no_vehicle_rate', 'acs_pct_white_non_hispanic', 'acs_pct_black_non_hispanic', 'acs_pct_hispanic']


In [23]:
# Ensure community_area is int
df_5e_ca['community_area'] = df_5e_ca['community_area'].astype(int)
df_311_ca['community_area'] = df_311_ca['community_area'].astype(int)
df_cps_ca['community_area'] = df_cps_ca['community_area'].astype(int)
df_acs_ca['community_area'] = df_acs_ca['community_area'].astype(int)

# Start with 5Essentials (has community_area_name)
df_merged = df_5e_ca.copy()

# Left join 311 data
df_merged = df_merged.merge(df_311_ca, on='community_area', how='left')

# Left join CPS data (drop duplicate community_area_name)
cps_cols = [c for c in df_cps_ca.columns if c != 'community_area_name']
df_merged = df_merged.merge(df_cps_ca[cps_cols], on='community_area', how='left')

# Left join ACS data
df_merged = df_merged.merge(df_acs_ca, on='community_area', how='left')

print(f"Rows: {len(df_merged)} community areas")
print(f"Columns: {len(df_merged.columns)}")

Rows: 77 community areas
Columns: 39


In [24]:
# Merge quality check
print("MERGE QUALITY CHECK:")
print(f"  Community areas with 5E data: {df_merged['n_schools_5e'].notna().sum()}/77")
print(f"  Community areas with 311 data: {df_merged['sr_total_requests'].notna().sum()}/77")
print(f"  Community areas with CPS data: {df_merged['cps_n_schools'].notna().sum()}/77")
print(f"  Community areas with ACS data: {df_merged['acs_poverty_rate'].notna().sum()}/77")

# Preview
display_cols = ['community_area', 'community_area_name', 'n_schools_5e',
                'sr_avg_median_ttc_hours', 'cps_student_attendance', 'acs_poverty_rate']
df_merged[display_cols].head(10)

MERGE QUALITY CHECK:
  Community areas with 5E data: 77/77
  Community areas with 311 data: 77/77
  Community areas with CPS data: 77/77
  Community areas with ACS data: 77/77


,community_area,community_area_name,n_schools_5e,sr_avg_median_ttc_hours,cps_student_attendance,acs_poverty_rate
0,1,ROGERS PARK,8,39.734542,87.8,0.1840
1,2,WEST RIDGE,9,198.747619,87.8,0.1929
2,3,UPTOWN,6,52.082995,87.8,0.1845
3,4,LINCOLN SQUARE,5,2636.383887,87.8,0.0930
4,5,NORTH CENTER,7,54.530971,87.8,0.0508
5,6,LAKE VIEW,11,86.360029,87.8,0.0827
6,7,LINCOLN PARK,8,723.079378,87.8,0.0852
7,8,NEAR NORTH SIDE,7,460.399048,87.8,0.1014
8,9,EDISON PARK,3,154.537720,87.8,0.0812
9,10,NORWOOD PARK,8,188.866678,87.8,0.0566


In [30]:
# Save final dataset
df_merged.to_csv('final_merged_5e_311_cps_acs.csv', index=False)


In [26]:
# Preview final dataset
print("Final merged dataset columns:")
print(df_merged.columns.tolist())

Final merged dataset columns:
['community_area', 'community_area_name', 'n_schools_5e', 'avg_leaders_score', 'avg_teachers_score', 'avg_families_score', 'avg_environment_score', 'avg_instruction_score', 'sr_total_requests', 'sr_total_closed', 'sr_avg_share_closed', 'sr_avg_median_ttc_hours', 'sr_avg_p75_ttc_hours', 'sr_avg_pct_closed_24h', 'sr_avg_pct_closed_7d', 'sr_avg_pct_closed_30d', 'cps_n_schools', 'cps_student_attendance', 'cps_teacher_attendance', 'cps_mobility_rate', 'cps_graduation_rate', 'cps_college_enrollment', 'acs_total_population', 'acs_poverty_rate', 'acs_median_household_income_weighted', 'acs_unemployment_rate', 'acs_pct_less_than_hs', 'acs_pct_bachelor_plus', 'acs_snap_rate', 'acs_public_assist_rate', 'acs_rent_burden_30p_rate', 'acs_rent_burden_50p_rate', 'acs_crowding_rate', 'acs_pct_renter', 'acs_vacancy_rate', 'acs_no_vehicle_rate', 'acs_pct_white_non_hispanic', 'acs_pct_black_non_hispanic', 'acs_pct_hispanic']


In [27]:
# Show first few rows
df_merged.head()

,community_area,community_area_name,n_schools_5e,avg_leaders_score,avg_teachers_score,avg_families_score,avg_environment_score,avg_instruction_score,sr_total_requests,sr_total_closed,...,acs_public_assist_rate,acs_rent_burden_30p_rate,acs_rent_burden_50p_rate,acs_crowding_rate,acs_pct_renter,acs_vacancy_rate,acs_no_vehicle_rate,acs_pct_white_non_hispanic,acs_pct_black_non_hispanic,acs_pct_hispanic
0,1,ROGERS PARK,8,41.166667,36.833333,43.166667,47.958333,46.875000,49646,48034,...,0.0446,0.4925,0.2351,0.0516,0.7133,0.0908,0.3604,0.4194,0.2312,0.2416
1,2,WEST RIDGE,9,60.296296,58.777778,59.555556,54.333333,56.481481,75476,73121,...,0.0353,0.5336,0.2907,0.0843,0.4938,0.0769,0.1499,0.3937,0.1171,0.2058
2,3,UPTOWN,6,51.777778,49.305556,47.361111,52.777778,54.305556,56086,54101,...,0.0350,0.4687,0.2188,0.0237,0.7072,0.0912,0.4285,0.5197,0.1901,0.1415
3,4,LINCOLN SQUARE,5,49.333333,51.000000,64.333333,52.466667,53.333333,53312,51497,...,0.0314,0.4328,0.2248,0.0200,0.5928,0.0618,0.2224,0.6232,0.0416,0.1778
4,5,NORTH CENTER,7,51.404762,51.190476,59.583333,61.761905,53.476190,53299,51362,...,0.0148,0.3052,0.1175,0.0024,0.4313,0.0870,0.1201,0.7334,0.0235,0.1185


In [31]:
# Save final dataset

df_merged.to_csv('final_merged_5e_311_cps_acs.csv', index=False)


## 8. Summary

### Crosswalk Methodology
- **Direct match (77%)**: School_ID from 5Essentials matched to 2011-12 CPS data
- **Geodesic match (23%)**: For newer schools, used GeoPy coordinates to find nearest school

### Output Files
- `5essentials_community_area_crosswalk.csv` — School-level crosswalk (school_id → community_area)
- `5essentials_by_community_area.csv` — Aggregated 5Essentials scores by community area
- `cps_by_community_area.csv` — Aggregated CPS outcomes by community area
- `final_merged_5e_311_cps_acs.csv` — Final merged dataset (77 community areas × 39 variables)

### Key Variables (39 total)
- **5Essentials (5)**: avg_leaders_score, avg_teachers_score, avg_families_score, avg_environment_score, avg_instruction_score
- **311 Metrics (8)**: sr_total_requests, sr_avg_median_ttc_hours, sr_avg_pct_closed_7d, etc.
- **CPS Outcomes (6)**: cps_student_attendance, cps_mobility_rate, cps_graduation_rate, cps_college_enrollment, etc.
- **ACS Demographics (17)**: acs_poverty_rate, acs_median_household_income_weighted, acs_unemployment_rate, etc.

In [32]:
# Final summary statistics

print(f"\nCrosswalk:")
print(f"  Total schools: {len(crosswalk)}")
print(f"  Direct matches: {(crosswalk['match_method'] == 'direct').sum()}")
print(f"  Geodesic matches: {(crosswalk['match_method'] == 'geodesic').sum()}")
print(f"  Unmatched: {crosswalk['community_area'].isna().sum()}")

print(f"\nFinal merged dataset:")
print(f"  Community areas: {len(df_merged)}")
print(f"  Variables: {df_merged.shape[1]}")


Crosswalk:
  Total schools: 648
  Direct matches: 498
  Geodesic matches: 147
  Unmatched: 3

Final merged dataset:
  Community areas: 77
  Variables: 39
